# Модуль 2. Урок 5: Оконные функции (Window Functions)

Оконные функции производят вычисления по группе строк, **не схлопывая строки** (в отличие от `GROUP BY`).

Синтаксис:
$$\text{FUNCTION}() \text{ OVER } (\text{PARTITION BY } col \text{ ORDER BY } col)$$


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"d:\vibe_app\main_data_analytics\course\datasets\ecommerce.db")

query = """
WITH monthly_sales AS (
    SELECT 
        SUBSTR(o.order_date, 1, 7) AS order_month,
        ROUND(SUM(oi.total_amount), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'Completed'
    GROUP BY order_month
)
SELECT 
    order_month,
    revenue,
    -- Нарастающий итог (Running Total)
    ROUND(SUM(revenue) OVER (ORDER BY order_month), 2) AS cumulative_revenue,
    -- Предыдущий месяц через LAG
    LAG(revenue, 1) OVER (ORDER BY order_month) AS prev_month_revenue,
    -- Темп прироста MoM %
    ROUND((revenue - LAG(revenue, 1) OVER (ORDER BY order_month)) * 100.0 / 
          LAG(revenue, 1) OVER (ORDER BY order_month), 2) AS mom_growth_pct
FROM monthly_sales
ORDER BY order_month;
"""
print(pd.read_sql(query, conn))
